# Análise k=1: MASTER, TFB e benchmarks

Fixa `janela_trading = k = 1`, em linha com a comparação de losses do paper **On Evaluating Loss Functions for Stock Ranking**.

No MASTER, normalmente `k = h`; logo `k=1` equivale a `pred_len/h=1`. No TFB, `k=1` pode coexistir com vários `pred_len`.


In [ ]:
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'utils':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.comparativo_metricas_global import comparar_global
OUT = ROOT / 'simulacoes' / 'comparativo_global_master_tfb'
OUT.mkdir(parents=True, exist_ok=True)
OUT


In [ ]:
dfs = comparar_global(
    base_dir=ROOT,
    output_dir='simulacoes/comparativo_global_master_tfb',
    top_n=30,
)
metricas = dfs['metricas'].copy()
print(f'Linhas carregadas: {len(metricas)}')
metricas.head()


In [ ]:
def simplificar_dataset(nome):
    nome = str(nome)
    if re.search(r'(__|_|-)log_returns?$', nome):
        return 'log_return'
    if re.search(r'(__|_|-)returns?$', nome):
        return 'return'
    if re.search(r'(__|_|-)prices?$', nome):
        return 'prices'
    return nome

def anexar_negativos_master(df):
    df = df.copy()
    if 'json_path' not in df.columns:
        return df
    df['output_dir'] = df['json_path'].apply(lambda p: str(Path(p).parent))
    neg_path = ROOT / 'simulacoes' / 'master_tfb_experimento' / 'acerto_negativos_master.csv'
    if not neg_path.exists():
        print(f'Aviso: {neg_path} não encontrado.')
        return df
    neg = pd.read_csv(neg_path)
    cols = [c for c in [
        'output_dir', 'taxa_acerto_negativos', 'mean_precision_negative',
        'n_pred_negativos', 'n_acertos_negativos', 'n_janelas_com_negativos'
    ] if c in neg.columns]
    out = df.merge(neg[cols].drop_duplicates('output_dir'), on='output_dir', how='left', suffixes=('', '_negcsv'))
    for c in cols:
        if c == 'output_dir':
            continue
        c2 = f'{c}_negcsv'
        if c2 in out.columns:
            out[c] = out[c].combine_first(out[c2]) if c in out.columns else out[c2]
            out = out.drop(columns=[c2])
    return out

metricas = anexar_negativos_master(metricas)
metricas['dataset_plot'] = metricas['dataset'].apply(simplificar_dataset) if 'dataset' in metricas.columns else np.nan
metricas['serie_modelo'] = metricas['grupo'].astype(str) + ' | ' + metricas['modelo'].astype(str)
metricas['janela_trading_num'] = pd.to_numeric(metricas['janela_trading'], errors='coerce')
k1 = metricas[metricas['janela_trading_num'].eq(1)].copy()
print(f'Linhas com k=1: {len(k1)}')
k1.head()


In [ ]:
# Cobertura k=1
display(k1.groupby('grupo').size().rename('n_resultados').reset_index())
display(k1.groupby(['grupo', 'modelo']).size().rename('n_resultados').reset_index().sort_values(['grupo', 'n_resultados'], ascending=[True, False]))


In [ ]:
metricas_k1 = [c for c in [
    'mean_spearman_ic', 'mean_precision_positive', 'mean_precision_negative',
    'auc_alta', 'auc_queda', 'auc_alta_media_janela', 'auc_queda_media_janela'
] if c in k1.columns]

chaves = ['grupo', 'dataset_plot', 'modelo', 'lookback', 'pred_len']
analise_k1 = (
    k1.groupby(chaves, dropna=False)[metricas_k1]
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
)
analise_k1.columns = [
    '_'.join([str(x) for x in col if x != '']) if isinstance(col, tuple) else col
    for col in analise_k1.columns
]
analise_k1 = analise_k1.sort_values(['mean_spearman_ic_median', 'mean_spearman_ic_mean'], ascending=False, na_position='last')
analise_k1.to_csv(OUT / 'analise_k1_ic_modelos.csv', index=False)
analise_k1.head(80)


In [ ]:
cols_top = [c for c in [
    'grupo', 'dataset_plot', 'modelo', 'lookback', 'pred_len',
    'mean_spearman_ic', 'mean_precision_positive', 'mean_precision_negative',
    'auc_alta', 'auc_queda', 'json_path'
] if c in k1.columns]
top_k1_ic = k1[cols_top].sort_values('mean_spearman_ic', ascending=False, na_position='last')
top_k1_ic.to_csv(OUT / 'top_k1_ic_configuracoes.csv', index=False)
top_k1_ic.head(80)


## Gráficos


In [ ]:
def plot_bar_median(df, metrica, titulo, top_n=30):
    if df.empty or metrica not in df.columns:
        print(f'Sem dados para {metrica}.')
        return None
    base = (df.dropna(subset=[metrica])
              .groupby(['grupo', 'modelo'], dropna=False)[metrica]
              .median().reset_index())
    base['serie'] = base['grupo'].astype(str) + ' | ' + base['modelo'].astype(str)
    base = base.sort_values(metrica, ascending=False).head(top_n).sort_values(metrica)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(base))))
    ax.barh(base['serie'], base[metrica])
    if 'spearman' in metrica:
        ax.axvline(0, linestyle='--', linewidth=1)
    ax.set_xlabel(metrica)
    ax.set_title(titulo)
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    return base

rank_ic = plot_bar_median(k1, 'mean_spearman_ic', 'Ranking de IC mediano - k=1')
rank_pos = plot_bar_median(k1, 'mean_precision_positive', 'Precisão positiva mediana - k=1')
rank_neg = plot_bar_median(k1, 'mean_precision_negative', 'Precisão negativa mediana - k=1')


In [ ]:
def plot_ic_por_param(df, param, top_n=12):
    if df.empty or param not in df.columns:
        print(f'Sem {param}.')
        return None
    base = df.dropna(subset=['mean_spearman_ic', param]).copy()
    if base.empty:
        print(f'Sem dados para {param}.')
        return None
    ordem = base.groupby('serie_modelo')['mean_spearman_ic'].median().sort_values(ascending=False).head(top_n).index
    base = base[base['serie_modelo'].isin(ordem)]
    agg = base.groupby(['serie_modelo', param], dropna=False)['mean_spearman_ic'].median().reset_index()
    fig, ax = plt.subplots(figsize=(12, 5))
    for serie, g in agg.groupby('serie_modelo'):
        g = g.sort_values(param)
        ax.plot(g[param], g['mean_spearman_ic'], marker='o', label=serie)
    ax.axhline(0, linestyle='--', linewidth=1)
    ax.set_xlabel(param)
    ax.set_ylabel('mean_spearman_ic mediano')
    ax.set_title(f'IC em k=1 por {param}')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()
    return agg

ic_por_lookback = plot_ic_por_param(k1, 'lookback')
ic_por_predlen = plot_ic_por_param(k1, 'pred_len')


## Saídas salvas

- `simulacoes/comparativo_global_master_tfb/analise_k1_ic_modelos.csv`
- `simulacoes/comparativo_global_master_tfb/top_k1_ic_configuracoes.csv`
